<a href="https://colab.research.google.com/github/techsatpal/4k_Video_Upscaler_Colab_-Real_ESRGAN-.ipynb/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU not detected. Enable T4 GPU."

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
import os
import sys
import pathlib
import subprocess
import shutil
import importlib

# 1. Install required dependencies (keep existing PyTorch)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "addict", "future", "lmdb", "pyyaml",
    "scipy", "yapf", "scikit-image", "ffmpeg-python"
], check=True)

# 2. Clone repositories if missing
os.chdir("/content")

if not os.path.isdir("/content/BasicSR/.git"):
    subprocess.run([
        "git", "clone",
        "https://github.com/XPixelGroup/BasicSR.git",
        "/content/BasicSR"
    ], check=True)

if not os.path.isdir("/content/Real-ESRGAN/.git"):
    subprocess.run([
        "git", "clone",
        "https://github.com/xinntao/Real-ESRGAN.git",
        "/content/Real-ESRGAN"
    ], check=True)

# 3. Fix BasicSR Python 3.13 version compatibility
basicsr_version = pathlib.Path(
    "/content/BasicSR/basicsr/version.py"
)

basicsr_version.write_text(
    '__version__ = "1.4.2"\n'
    '__gitsha__ = "unknown"\n'
)

# 4. Fix Real-ESRGAN version file
realesrgan_version = pathlib.Path(
    "/content/Real-ESRGAN/realesrgan/version.py"
)

realesrgan_version.write_text(
    '__version__ = "0.3.0"\n'
)

# 5. Fix torchvision compatibility in BasicSR
degradations = pathlib.Path(
    "/content/BasicSR/basicsr/data/degradations.py"
)

if degradations.exists():
    code = degradations.read_text()

    code = code.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale"
    )

    degradations.write_text(code)

# 6. Add repositories to Python path
sys.path.insert(0, "/content/BasicSR")
sys.path.insert(0, "/content/Real-ESRGAN")

# Clear import caches
for name in list(sys.modules):
    if name == "basicsr" or name.startswith("basicsr."):
        del sys.modules[name]

importlib.invalidate_caches()

# 7. Download model if missing
weights_dir = "/content/Real-ESRGAN/weights"
os.makedirs(weights_dir, exist_ok=True)

model_path = os.path.join(
    weights_dir, "RealESRGAN_x4plus.pth"
)

if not os.path.exists(model_path):
    subprocess.run([
        "wget", "-q",
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "-P", weights_dir
    ], check=True)

# 8. Install/check FFmpeg
if shutil.which("ffmpeg") is None:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "ffmpeg"],
        check=True
    )

# 9. Verify imports
import basicsr
import realesrgan
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

print("\nSETUP SUCCESSFUL")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BasicSR:", basicsr.__file__)
print("Real-ESRGAN:", realesrgan.__file__)
print("Model exists:", os.path.exists(model_path))
print("FFmpeg:", shutil.which("ffmpeg"))


SETUP SUCCESSFUL
Python: 3.13.16
PyTorch: 2.11.0+cu130
GPU: Tesla T4
BasicSR: /content/BasicSR/basicsr/__init__.py
Real-ESRGAN: /content/Real-ESRGAN/realesrgan/__init__.py
Model exists: True
FFmpeg: /usr/bin/ffmpeg


In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = "/content/" + next(iter(uploaded))

print("Video selected:", video_path)

Saving Portable LED Selfie Light for Phone.webm to Portable LED Selfie Light for Phone (5).webm
Video selected: /content/Portable LED Selfie Light for Phone (5).webm


# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [ ]:
import os
import cv2
import subprocess
from google.colab import files

video_path = video_path
output_dir="/content/" #@param{type:"string"}
resolution = "FHD (1920 x 1080)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

assert os.path.exists(video_path), "Video file does not exist"
video_capture = cv2.VideoCapture(video_path)
video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

final_width = None
final_height = None
aspect_ratio = float(video_width/video_height)

# Get output resolutions
match resolution:
  case "FHD (1920 x 1080)":
    final_width=1920
    final_height=1080
  case "2k (2560 x 1440)":
    final_width=2560
    final_height=1440
  case "4k (3840 x 2160)":
    final_width=3840
    final_height=2160
  case "2 x original":
    final_width=2*video_width
    final_height=2*video_height
  case "3 x original":
    final_width=3*video_width
    final_height=3*video_height
  case "4 x original":
    final_width=4*video_width
    final_height=4*video_height

if aspect_ratio == 1.0 and "original" not in resolution:
  final_height = final_width

if aspect_ratio < 1.0 and "original" not in resolution:
  temp = final_width
  final_width = final_height
  final_height = temp

scale_factor = max(final_width/video_width, final_height/video_height)
isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

# scale_factor needs to be even
while isEven == False:
  scale_factor += 0.01
  isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

print(f"Upscaling from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")
# Convert input to MP4 for Real-ESRGAN compatibility
if video_path.lower().endswith(".webm"):

    converted_video_path = os.path.splitext(video_path)[0] + "_converted.mp4"

    command = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-map", "0:v:0",
        "-map", "0:a?",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        converted_video_path
    ]

    subprocess.run(command, check=True)

    video_path = converted_video_path

print("Video converted successfully:", video_path, flush=True)

# Run Real-ESRGAN video inference
env = os.environ.copy()
env["PYTHONPATH"] = "/content/BasicSR:/content/Real-ESRGAN:" + env.get("PYTHONPATH", "")

command = [
    "python",
    "/content/Real-ESRGAN/inference_realesrgan_video.py",
    "-n", model,
    "-i", video_path,
    "-o", output_dir,
    "--outscale", str(scale_factor)
]

print("Starting Real-ESRGAN inference...", flush=True)

subprocess.run(command, env=env, check=True)

video_name_with_ext = os.path.basename(video_path)
video_name = os.path.splitext(video_name_with_ext)[0]
upscaled_video_path = os.path.join(output_dir, f"{video_name}_out.mp4")
assert os.path.exists(upscaled_video_path), "Real-ESRGAN failed to generate the upscaled video."
final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
final_video_path = os.path.join(output_dir, final_video_name)

# crop to fit
if "original" not in resolution:
  print("Cropping to fit...")
  command = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' -c:v h264_nvenc -filter:v  'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
  subprocess.run(command,shell=True)
else:
  # final video path = upscaled video path
  command = f"cp '{upscaled_video_path}' '{final_video_path}'"
  subprocess.run(command,shell=True)

print(f"Upscaled video saved to: {final_video_path}")

# # save to drive
# if mount_drive:
#   drive_folder = "MyDrive/Upscaled Videos (REAL-ESRGAN)"
#   save_directory_drive = f"/content/gdrive/{drive_folder}"
#   os.makedirs(save_directory_drive, exist_ok=True)

#   command = f"cp '{final_video_path}' '{save_directory_drive}/{final_video_name}'"
#   subprocess.run(command,shell=True)
#   print(f"Saved to drive: /{drive_folder}/{final_video_name}" )
# Download final upscaled video

# Check audio codec
result = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_name",
        "-of", "default=noprint_wrappers=1:nokey=1",
        final_video_path
    ],
    capture_output=True,
    text=True
)

audio_codec = result.stdout.strip()

print("Audio codec:", audio_codec if audio_codec else "No audio track")

if audio_codec and audio_codec != "aac":
    aac_video_path = final_video_path.replace(".mp4", "_aac.mp4")

    subprocess.run([
        "ffmpeg", "-y", "-i", final_video_path,
        "-c:v", "copy",
        "-c:a", "aac", "-b:a", "192k",
        aac_video_path
    ], check=True)

    os.replace(aac_video_path, final_video_path)

files.download(final_video_path)

files.download(final_video_path)
!rm "{upscaled_video_path}"

Upscaling from 336x608 to 672x1216, scale_factor=2.0
Upscaled video saved to: /content/Portable LED Selfie Light for Phone (5)_converted_upscaled_672_1216.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()